# gap_bench — 로컬 실험 노트북

**폴더 구조:** 이 노트북은 `gap_bench/` 루트에서 열고, 코드는 `src/`, 결과는 `results/`, 트레이스는 `traces/`.  
**요구사항:** Python 3.10+, RAM 8GB+, 디스크 3GB. GPU 불필요.  
**재개:** 중단 후 [5]만 다시 실행 (완료 슬라이스 자동 skip).


## [1] 환경 확인 및 의존성


In [1]:
import sys, os
print('Python:', sys.version.split()[0]); print('작업 폴더:', os.getcwd())
assert os.path.exists('src/formulation.py'), 'gap_bench 루트에서 노트북을 여세요'
sys.path.insert(0, 'src')
%pip -q install pulp highspy pyarrow pandas


Python: 3.12.8
작업 폴더: D:\GoogleDrive_dkimrok\gap_bench
Note: you may need to restart the kernel to use updated packages.


## [2] 환경 검증 — **전 항목 ✔ 확인 후 진행**


In [2]:
import subprocess, sys, os
env = {**os.environ, 'PYTHONIOENCODING': 'utf-8'}   # Windows cp949 대응
r = subprocess.run([sys.executable, 'src/toy_test.py'],
                   capture_output=True, text=True, encoding='utf-8', env=env)
print(r.stdout, r.stderr)
assert r.returncode == 0, 'toy_test 실패 — DEVLOG 확인'


[brute force] optimal total latency = 27, admit = {0: 3, 1: 0, 2: 0, 3: 1, 4: 3}
[MILP] status=Optimal obj=27.0 bound=27.0 admit={0: 3, 1: 0, 2: 0, 3: 1, 4: 3}
[check] MILP == brute force, replay violations = 0 
[fcfs] latency=42 ttft=26 gap=+55.6%
[sjf] latency=27 ttft=11 gap=+0.0%
[warm start] incumbent=27.0 (<= fcfs 42) 
 


## [3] 트레이스 준비 (최초 1회, ~2GB 다운로드, 15–20분)


In [3]:
import os, subprocess, sys, glob
env = {**os.environ, 'PYTHONIOENCODING': 'utf-8'}
if not os.path.exists('traces/conv.parquet'):
    subprocess.run([sys.executable, 'src/prep_trace.py'], check=True, env=env)
    for f in glob.glob('traces/azure_llm_2024_*.csv'):
        os.remove(f)
else:
    print('parquet 이미 존재 — 건너뜀')
print(os.listdir('traces'))


parquet 이미 존재 — 건너뜀
['code.parquet', 'conv.parquet']


## [4] 실행 파라미터


In [4]:
PILOT = True        # False = 본그리드 (50슬라이스, n=400) — 밤새 실행 권장
TIME_LIMIT = 900    # 슬라이스당 MILP 초 (로컬은 1800까지 상향 가능)


## [5] 실행 (재개 가능)


In [5]:
import run_slices
run_slices.main(pilot=PILOT, time_limit=TIME_LIMIT)


grid: 6 slices (PILOT), done: 3, TL=900s
[skip] conv_r0.3_o0_n200
[skip] conv_r0.3_o20_n200
[skip] conv_r0.6_o0_n200
[done] conv_r0.6_o20_n200: fcfs_gap=[+46.9%,+60.7%] milp_gap=8.6% t=940s
[done] conv_r0.9_o0_n200: fcfs_gap=[+24.5%,+44.5%] milp_gap=13.9% t=938s
[done] conv_r0.9_o20_n200: fcfs_gap=[+23.3%,+35.6%] milp_gap=9.0% t=937s


## [6] 점검 (warm_applied / milp_gap / 소요시간)


In [6]:
import pandas as pd
df = pd.read_csv('results/results.csv')
df['fcfs_gap_lo'] = (df.fcfs_lat - df.milp_incumbent) / df.milp_incumbent
df['fcfs_gap_hi'] = (df.fcfs_lat - df.milp_bound) / df.milp_bound
df['sjf_gap_lo']  = (df.sjf_lat  - df.milp_incumbent) / df.milp_incumbent
df['sjf_gap_hi']  = (df.sjf_lat  - df.milp_bound) / df.milp_bound
ok = bool(df.warm_applied.astype(bool).all())
print('1) warm_applied 전부 True? ->', ok, '' if ok else '*** 문제: DEVLOG 참조 ***')
print(f'2) milp_gap: mean={df.milp_gap.mean():.1%}  max={df.milp_gap.max():.1%}  (목표: max 10% 이하)')
print(f'3) 슬라이스당: mean={df.solve_s.mean()/60:.1f}분  ->  본그리드 50슬라이스 예상 {df.solve_s.mean()*50/3600:.1f}h')
cols = ['slice_id','fcfs_gap_lo','fcfs_gap_hi','sjf_gap_lo','sjf_gap_hi','milp_gap','milp_status','solve_s']
df[cols]


1) warm_applied 전부 True? -> True 
2) milp_gap: mean=11.0%  max=15.4%  (목표: max 10% 이하)
3) 슬라이스당: mean=15.8분  ->  본그리드 50슬라이스 예상 13.1h


,slice_id,fcfs_gap_lo,fcfs_gap_hi,sjf_gap_lo,sjf_gap_hi,milp_gap,milp_status,solve_s
0,conv_r0.3_o0_n200,0.405328,0.660175,0.076454,0.271662,0.153506,Time limit reached,967.9
1,conv_r0.3_o20_n200,0.483961,0.607054,0.243699,0.346863,0.076596,Time limit reached,939.6
2,conv_r0.6_o0_n200,0.642767,0.853641,0.246484,0.406489,0.113762,Time limit reached,953.5
3,conv_r0.6_o20_n200,0.468522,0.607344,0.271949,0.392188,0.086367,Time limit reached,939.5
4,conv_r0.9_o0_n200,0.244995,0.445260,0.136328,0.319113,0.138567,Time limit reached,938.4
5,conv_r0.9_o20_n200,0.233156,0.355836,0.218291,0.339492,0.090483,Time limit reached,936.8


## [7] 본실험 전환: [4]에서 `PILOT = False` 후 [5] 재실행

장시간 실행은 터미널이 안정적: `python src/run_slices.py --main --tl 900` (절전 해제 필수).


## [부록] 결과 원장 백업


In [7]:
import time, shutil, os
os.makedirs('results/backup', exist_ok=True)
dst = f"results/backup/results_{time.strftime('%m%d_%H%M')}.csv"
shutil.copy('results/results.csv', dst)
print('backup ->', dst)


backup -> results/backup/results_0805_1127.csv
